#  Phil Job Applier Agent - Rachel Briskman & Sanila Chowdhury (Daedalus Devs)

## Team Report #1 (03/24) - Created a function that takes in user's resume and parse it to output the extracted info & uses Skyvern to see how the library works for filling in the fields of a job website

<img src="AI_AGENT_WORKFLOW.png" width="700">

### Installs and Imports

In [ ]:

%pip install chromadb
# %pip install "skyvern==1.0.24"
%pip install skyvern
# %playwright install --with-deps chromium
# %playwright install chromium
%pip install langchain langchain-huggingface
%pip install python-dotenv

print ("Done installing")

In [ ]:
#installing dependencies for browser-use (an alternative to skyvern)
%pip install browser-use playwright
%pip install -U langchain-google-genai
%pip install -U browser-use
# %playwright install chromium

### Import API keys from .env file
#### (Note: This is for local use. Colab imports keys differently)

In [18]:
import os
from dotenv import load_dotenv
import platform
import sys


#Sanila needs to set the working directory explicity (macOS) but it's not necessary on Windows or WSL.
#added this macro so we don't have to manually comment/uncommment os.chdir() every time.

IS_MAC = platform.system() == "Darwin"

if IS_MAC:
    os.chdir("/Users/sanilachowdhury/Desktop/ai_agents")
    print("Running on macOS")
else:
    print("Not running on macOS, current directory:", os.getcwd())


load_dotenv()


gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")


os.environ["ENABLE_AUTH"] = "false"

# from skyvern.forge.sdk import SkyvernForge
from skyvern import Skyvern
import nest_asyncio
import asyncio

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# 2. Map it to the environment variables Skyvern/LiteLLM looks for
os.environ["GEMINI_API_KEY"] = gemini_key
# os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# Optional: Force Skyvern to use a specific model
os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# print("Keys loaded and environment configured.")

Running on macOS
Key check: FOUND


### A website we're feeding into our agent to fill in the field (this website is used for testing purpose)

In [2]:
url = "https://demo.applitools.com/"

### Agent takes in the filepath where the user's resume is stored and we prompt the agent to extract the info in a JSON that easy for both LLM and humans to read

In [29]:
import asyncio
import nest_asyncio
import os
from google import genai
from IPython.display import display, Markdown

nest_asyncio.apply()
client_genai = genai.Client(api_key=gemini_key)

def parse_resume(file_path, prompt="Please parse this resume and return its contents in a clear, readable format. Use sections with headers (like Contact, Summary, Experience, Education, Skills) and plain text or bullet points — no JSON or code blocks."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    Caches the result so Gemini is only called once per resume.
    """
    cache_path = file_path.replace(".pdf", "_cached.json")

    # delete old cache if it exists to force re-parse with new format
    # if os.path.exists(cache_path):
    #     os.remove(cache_path)
    #     print(f"Cleared old cache at {cache_path}.")

    if os.path.exists(cache_path):
        print(f"Using cached resume data from {cache_path}...")
        with open(cache_path, "r") as f:
            return f.read()

    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

    # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    # save to cache so we don't call Gemini again
    with open(cache_path, "w") as f:
        f.write(response.text)
    print(f"Resume parsed and cached to {cache_path}")

    return response.text

# call the function
result = parse_resume("content/jakes-resume.pdf")
display(Markdown(f"<div style='font-size: 14px'>{result}</div>"))

Using cached resume data from content/jakes-resume_cached.json...


<div style='font-size: 14px'>Here's a breakdown of the resume:

**Contact**
*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   August 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   August 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
        *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems.
        *   Developed a full-stack web application using Flask, React, PostgreSQL, and Docker to analyze GitHub data.
        *   Explored ways to visualize GitHub collaboration in a classroom setting.
*   **Information Technology Support Specialist**
    *   Southwestern University
    *   Georgetown, TX
    *   September 2018 - Present
        *   Communicate with managers to set up campus computers used on campus.
        *   Assess and troubleshoot computer problems brought by students, faculty, and staff.
        *   Maintain upkeep of computers, classroom equipment, and 200 printers across campus.
*   **Artificial Intelligence Research Assistant**
    *   Southwestern University
    *   Georgetown, TX
    *   May 2019 - July 2019
        *   Explored methods to generate video game dungeons based off of *The Legend of Zelda*.
        *   Developed a game in Java to test the generated dungeons.
        *   Contributed 50K+ lines of code to an established codebase via Git.
        *   Conducted a human subject study to determine which video game dungeon generation technique is enjoyable.
        *   Wrote an 8-page paper and gave multiple presentations on-campus.
        *   Presented virtually to the World Conference on Computational Intelligence.

**Projects**

*   **Gitlytics**
    *   Python, Flask, React, PostgreSQL, Docker
    *   June 2020 - Present
        *   Developed a full-stack web application using Flask serving a REST API with React as the frontend.
        *   Implemented GitHub OAuth to get data from user's repositories.
        *   Visualized GitHub data to show collaboration.
        *   Used Celery and Redis for asynchronous tasks.
*   **Simple Paintball**
    *   Spigot API, Java, Maven, TravisCI, Git
    *   May 2018 - May 2020
        *   Developed a Minecraft server plugin to entertain kids during free time for a previous job.
        *   Published plugin to websites gaining 2K+ downloads and an average 4.5/5-star review.
        *   Implemented continuous delivery using TravisCI to build the plugin upon a new release.
        *   Collaborated with Minecraft server administrators to suggest features and get feedback about the plugin.

**Technical Skills**

*   **Languages:** Java, Python, C/C++, SQL (Postgres), JavaScript, HTML/CSS, R
*   **Frameworks:** React, Node.js, Flask, JUnit, WordPress, Material-UI, FastAPI
*   **Developer Tools:** Git, Docker, TravisCI, Google Cloud Platform, VS Code, Visual Studio, PyCharm, IntelliJ, Eclipse
*   **Libraries:** pandas, NumPy, Matplotlib</div>

## Team Report #2 (03/31) - Created human in the loop for resume info verification, set up ChromaDB, and opened a window of the job website (dummy url) and filled in a few fields.
### (Replaced Skyvern with Browser_Use). This code will not run on Colab, only locally.

<img src="AI_AGENT_WORKFLOW_REPORT_2.png" width="800">

### Developed a human in the loop where agent takes in the extracted info and ask the user if the info is correct. Until the user is satisifed, the agent will keep updating the extracted info based on user's feedback.

In [32]:
from IPython.display import display, Markdown

def verify_resume_info(extracted_info):
    current_info = extracted_info
    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        display(Markdown(f"<div style='font-size: 16px'>{current_info}</div>"))
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()
        print(f"Is this information correct? (y/n): {user_confirm}")

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            break
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()
            print(f"What needs to be corrected or added? Please describe: {correction}")

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}
The user wants the following correction or addition:
{correction}
Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")
    return current_info


# call the function to store the extracted info
result = parse_resume("content/jakes-resume.pdf")

# run verify_resume_info to verify and correct
final_info = verify_resume_info(result)

Using cached resume data from content/jakes-resume_cached.json...

EXTRACTED RESUME INFO:


<div style='font-size: 16px'>Here's a breakdown of the resume:

**Contact**
*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   August 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   August 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
        *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems.
        *   Developed a full-stack web application using Flask, React, PostgreSQL, and Docker to analyze GitHub data.
        *   Explored ways to visualize GitHub collaboration in a classroom setting.
*   **Information Technology Support Specialist**
    *   Southwestern University
    *   Georgetown, TX
    *   September 2018 - Present
        *   Communicate with managers to set up campus computers used on campus.
        *   Assess and troubleshoot computer problems brought by students, faculty, and staff.
        *   Maintain upkeep of computers, classroom equipment, and 200 printers across campus.
*   **Artificial Intelligence Research Assistant**
    *   Southwestern University
    *   Georgetown, TX
    *   May 2019 - July 2019
        *   Explored methods to generate video game dungeons based off of *The Legend of Zelda*.
        *   Developed a game in Java to test the generated dungeons.
        *   Contributed 50K+ lines of code to an established codebase via Git.
        *   Conducted a human subject study to determine which video game dungeon generation technique is enjoyable.
        *   Wrote an 8-page paper and gave multiple presentations on-campus.
        *   Presented virtually to the World Conference on Computational Intelligence.

**Projects**

*   **Gitlytics**
    *   Python, Flask, React, PostgreSQL, Docker
    *   June 2020 - Present
        *   Developed a full-stack web application using Flask serving a REST API with React as the frontend.
        *   Implemented GitHub OAuth to get data from user's repositories.
        *   Visualized GitHub data to show collaboration.
        *   Used Celery and Redis for asynchronous tasks.
*   **Simple Paintball**
    *   Spigot API, Java, Maven, TravisCI, Git
    *   May 2018 - May 2020
        *   Developed a Minecraft server plugin to entertain kids during free time for a previous job.
        *   Published plugin to websites gaining 2K+ downloads and an average 4.5/5-star review.
        *   Implemented continuous delivery using TravisCI to build the plugin upon a new release.
        *   Collaborated with Minecraft server administrators to suggest features and get feedback about the plugin.

**Technical Skills**

*   **Languages:** Java, Python, C/C++, SQL (Postgres), JavaScript, HTML/CSS, R
*   **Frameworks:** React, Node.js, Flask, JUnit, WordPress, Material-UI, FastAPI
*   **Developer Tools:** Git, Docker, TravisCI, Google Cloud Platform, VS Code, Visual Studio, PyCharm, IntelliJ, Eclipse
*   **Libraries:** pandas, NumPy, Matplotlib</div>

Is this information correct? (y/n): y

Great! Moving forward with this information.


### Setting up ChromaDB which will store the correct parsed info of user's resume

In [33]:
import chromadb
import os
import hashlib
from dotenv import load_dotenv

# debug - check if values are loaded
print(f"CHROMA_HOST: {os.getenv('CHROMA_HOST')}")
print(f"CHROMA_TENANT: {os.getenv('CHROMA_TENANT')}")
print(f"CHROMA_DATABASE: {os.getenv('CHROMA_DATABASE')}")
print(f"CHROMA_API_KEY: {'FOUND' if os.getenv('CHROMA_API_KEY') else 'MISSING'}")

chroma_client = chromadb.HttpClient(
    ssl=True,
    host=os.getenv("CHROMA_HOST"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    headers={"x-chroma-token": os.getenv("CHROMA_API_KEY")}
)
collection = chroma_client.get_or_create_collection(name="resumes")
print(f"Connected to ChromaDB! Total resumes stored: {collection.count()}")

def store_resume_in_chromadb(resume_text, file_path):
    doc_id = hashlib.md5(file_path.encode()).hexdigest()
    collection.upsert(
        documents=[resume_text],
        ids=[doc_id],
        metadatas=[{"file_path": file_path}]
    )
    print(f"Resume stored! ID: {doc_id}")
    return doc_id

# store the verified resume
doc_id = store_resume_in_chromadb(final_info, "content/jakes-resume.pdf")
print("Resume stored successfully!")

CHROMA_HOST: api.trychroma.com
CHROMA_TENANT: 7eab8b90-a03c-4bc3-9704-e5e4f8a3f82b
CHROMA_DATABASE: phil-job-agent
CHROMA_API_KEY: FOUND
Connected to ChromaDB! Total resumes stored: 1
Resume stored! ID: b7a0358df030937c049bb9754117f552
Resume stored successfully!


## Team Report #3 (04/14) - TBD

### Prompt user if they would like to upload a new resume if it's their first time. Otherwise, it will grab the resume that's stored in ChromaDB

In [ ]:
def apply_to_job_with_agent():
    resume_exists = input("Would you like to upload a new resume? Enter y if this is your first time (y/n): ")

    while(resume_exists != "y" and resume_exists != "n"):
        resume_exists = input("Invalid input. Please enter y or n:")

    if resume_exists == "y":
        file_path = input("Please enter the path to your resume PDF: ").strip()
        extracted_info = parse_resume(file_path)
        verified_info = verify_resume_info(extracted_info)
        store_resume_in_chromadb(verified_info, file_path)
    else:
        print("Using existing resume data. Moving on to job application...")


    url = input("Please enter the URL of the job application you are applying to: ").strip()

    #TODO: call the fill_application() with the url and the verified resume info (final_info)
    #fill_application(url, final_info) function does not exist yet

apply_to_job_with_agent()


Using existing resume data. Moving on to job application...


### Created a popup window of the job website that user will feed into Phil

In [ ]:
#NOTE: This uses browser-use instead of skyvern. This cell works but it does not promp the user for addtional info when needed. See next cell.
import os
import asyncio
from dotenv import load_dotenv
from browser_use import Agent, BrowserProfile, BrowserSession
from browser_use.llm import ChatGoogle

load_dotenv()

url = "https://demo.applitools.com/"

async def run_locally():
    llm = ChatGoogle(
        # model="gemini-2.5-flash-lite",
        model = "gemini-2.0-flash",
        # model = "gemini-1.5-flash",
        api_key=os.getenv("GEMINI_API_KEY")
    )

    browser_profile = BrowserProfile(headless=False, keep_alive=True)
    browser_session = BrowserSession(browser_profile=browser_profile)

    agent = Agent(
        task=f"Go to {url} and fill in 'test_user' and 'password123' BUT DO NOT PRESS SUBMIT OR LOGIN. Return DONE when complete",
        llm=llm,
        browser_session=browser_session,
        max_steps=5,
        use_vision=False,              # biggest saving — no screenshots sent to LLM
        max_actions_per_step=3,        # fewer actions considered per step
        include_attributes=[           # only send useful HTML attributes, not all of them
            'id', 'name', 'type', 'placeholder', 'aria-label'
        ],
        max_history_items=6,           # shorter memory = fewer tokens per step
        message_compaction=True,       # compress old messages automatically
        max_clickable_elements_length=20, 
    )

    result = await agent.run()
    if(result.final_result):
        print("Agent's final result: ", result.final_result)
    else:
        print("Agent did not return a final result.")

await run_locally()

2026-04-07T23:55:40.698987Z [error    ]  💥 API call failed after 0.35s: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 19.316699003s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'h

Agent's final result:  <bound method AgentHistoryList.final_result of AgentHistoryList(all_results=[ActionResult(is_done=False, success=None, judgement=None, error=None, attachments=None, images=None, long_term_memory='Found initial url and automatically loaded it. Navigated to https://demo.applitools.com/', extracted_content='🔗 Navigated to https://demo.applitools.com/', include_extracted_content_only_once=False, metadata=None, include_in_memory=False), ActionResult(is_done=False, success=None, judgement=None, error="429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\\n* Quota exceeded for metric: generati

: 

### An attempt at implementing "Gather extra info from user" while the browser window is open.
### Works but incomplete so commented for now. 

In [ ]:
# from browser_use import Agent, BrowserProfile, BrowserSession
# from browser_use.llm import ChatGoogle

# async def run_job_application():
#     llm = ChatGoogle(model="gemini-2.5-flash-lite", api_key=os.getenv("GEMINI_API_KEY"))
    
#     browser_profile = BrowserProfile(headless=False, keep_alive=True)
#     browser_session = BrowserSession(browser_profile=browser_profile)

#     # Phase 1: fill what we know
#     agent = Agent(
#         task=f"""Go to {url} and fill in the username with a_cool_username_123. 
#         If you encounter fields you don't have info for, stop and list what's missing. 
#         Do NOT submit.""",
#         llm=llm,
#         browser_session=browser_session,
#         use_vision=False,  # reduces tokens significantly
#     )
#     result = await agent.run()
#     print("Agent stopped. Result:", result)


#     #NOTE: Might be better to put this in a second cell or loop this?
#     # Phase 2: ask user for missing info
#     missing = input("What additional info do you want to provide? ")

#     # Phase 3: continue with same browser session (no reload, no extra screenshots)
#     agent2 = Agent(
#         task=f"Fill in the remaining fields with: {missing}. Do NOT submit.",
#         llm=llm,
#         browser_session=browser_session,  # reuse same session!
#     )
#     result2 = await agent2.run()
#     print("Done:", result2)

# await run_job_application()

## Plans for spring break:
* Implementing a function where our agent to fill in the fields from the job website using Skyvern (we have started this process but currently experiencing errors)
* Implementing a loop for our agent to handle when it comes to short-answer response
  * Have a prompt that will tell the LLM to generate answer based on the context given and the info it extracted from user's resume
  * Have a loop where as long as the answer is too vague, keep asking user for more info and implement a function for LLM to do web search (depending on the short answer question)
  * Have LLM generate a short answer response and fill out the field with Skyvern
* IF WE FIND TIME -> Have an evalulation where user can give the LLM feedback at the end of the overall performance and store that in ChromaDB to help Phil improve